# YOLOv8L Inference 파라미터 튜닝

**목적**: 모델 weight 변경 없이 후처리 파라미터(conf, iou, imgsz)의 최적 조합을 찾는다.

| 항목 | 설정 |
|---|---|
| 모델 | yolov8l.pt (weight 고정) |
| 검증 데이터셋 | COCO128 (Ground Truth 기반) |
| 평가 클래스 | MVP 15 클래스 |
| 튜닝 파라미터 | conf (0.25~0.8), iou (0.3~0.8), imgsz (320~960) |

**런타임 → GPU(T4) 로 변경 후 실행하세요!**

## STEP 1. 환경 설정

In [ ]:
!pip install -q ultralytics

import warnings
warnings.filterwarnings('ignore')

import time
import torch
import pandas as pd
from ultralytics import YOLO
from itertools import product

# GPU 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✔ GPU 사용: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ CPU 모드 — 런타임 > GPU로 변경하세요!')

print(f'PyTorch: {torch.__version__}')

## STEP 2. 모델 로드 & 설정

In [ ]:
# 모델 로드 (자동 다운로드)
model = YOLO('yolov8l.pt')
print(f'✔ 모델 로드 완료: yolov8l.pt')
print(f'✔ 클래스 수: {len(model.names)}개')

# MVP 15 클래스 정의
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]
MVP_CLASS_NAMES = {
    16: 'dog', 26: 'handbag', 39: 'bottle', 41: 'cup', 45: 'bowl',
    53: 'apple', 56: 'chair', 57: 'couch', 59: 'bed', 60: 'dining table',
    62: 'tv', 63: 'laptop', 65: 'remote', 67: 'cell phone', 72: 'refrigerator'
}

print(f'✔ MVP 15 클래스: {list(MVP_CLASS_NAMES.values())}')

## STEP 3. 탐색 범위 설정

In [ ]:
# ============================================
# 탐색할 파라미터 범위
# ============================================
CONF_RANGE  = [0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8]
IOU_RANGE   = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
IMGSZ_RANGE = [320, 480, 640, 960]

total_combos = len(CONF_RANGE) * len(IOU_RANGE) * len(IMGSZ_RANGE)

print(f'conf  탐색: {CONF_RANGE}')
print(f'iou   탐색: {IOU_RANGE}')
print(f'imgsz 탐색: {IMGSZ_RANGE}')
print(f'\n총 조합 수: {total_combos}개')
print(f'예상 소요: GPU 기준 약 {total_combos * 15 // 60}~{total_combos * 25 // 60}분')

## STEP 4. 그리드 서치 실행

모든 조합에 대해 COCO128 validation을 실행하고 Precision/Recall/F1을 기록합니다.

In [ ]:
results = []
t_total = time.time()

for idx, (conf, iou, imgsz) in enumerate(product(CONF_RANGE, IOU_RANGE, IMGSZ_RANGE), 1):
    t_start = time.time()

    # COCO128 validation (MVP 15 클래스 필터)
    try:
        metrics = model.val(
            data='coco128.yaml',
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            classes=MVP_CLASS_IDS,
            batch=16,
            device=device,
            verbose=False,
            half=True,  # GPU FP16 가속
        )

        p = float(metrics.box.mp)
        r = float(metrics.box.mr)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        elapsed = time.time() - t_start

        results.append({
            'conf': conf,
            'iou': iou,
            'imgsz': imgsz,
            'precision': round(p, 4),
            'recall': round(r, 4),
            'f1': round(f1, 4),
            'time_s': round(elapsed, 1),
        })

        # 진행률 표시
        pct = idx / total_combos * 100
        total_elapsed = time.time() - t_total
        eta = (total_elapsed / idx) * (total_combos - idx)
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))

        print(
            f'[{bar}] {idx}/{total_combos} ({pct:.0f}%)  '
            f'conf={conf} iou={iou} imgsz={imgsz}  '
            f'F1={f1:.4f}  P={p:.4f}  R={r:.4f}  '
            f'({elapsed:.1f}s)  ETA: {eta:.0f}s'
        )

    except Exception as e:
        print(f'[!] 실패: conf={conf} iou={iou} imgsz={imgsz} — {e}')
        continue

total_time = time.time() - t_total
print(f'\n============================================')
print(f'  그리드 서치 완료!  총 {len(results)}개 조합  |  소요: {total_time:.0f}s ({total_time/60:.1f}분)')
print(f'============================================')

## STEP 5. 결과 분석 — TOP 10 & 최적 조합

In [ ]:
df = pd.DataFrame(results)

# F1 기준 정렬
df_sorted = df.sort_values('f1', ascending=False).reset_index(drop=True)
df_sorted.index = df_sorted.index + 1  # 1부터 시작
df_sorted.index.name = '순위'

print('\n' + '='*70)
print('  TOP 10 파라미터 조합 (F1 Score 기준)')
print('='*70)
print(df_sorted.head(10).to_string())

# 최적 조합
best = df_sorted.iloc[0]
print(f'\n{"="*70}')
print(f'  ★ 최적 조합 ★')
print(f'  conf={best["conf"]}  iou={best["iou"]}  imgsz={int(best["imgsz"])}')
print(f'  Precision={best["precision"]}  Recall={best["recall"]}  F1={best["f1"]}')
print(f'{"="*70}')

# 기존 기본값과 비교
baseline = df[(df['conf']==0.25) & (df['iou']==0.7) & (df['imgsz']==640)]
if not baseline.empty:
    bl = baseline.iloc[0]
    f1_diff = best['f1'] - bl['f1']
    print(f'\n  기본값 (conf=0.25, iou=0.7, imgsz=640)')
    print(f'  Precision={bl["precision"]}  Recall={bl["recall"]}  F1={bl["f1"]}')
    print(f'\n  → F1 개선: {bl["f1"]} → {best["f1"]} ({f1_diff:+.4f}, {f1_diff/bl["f1"]*100:+.1f}%)')

## STEP 6. 파라미터별 영향 분석

In [ ]:
print('\n' + '='*50)
print('  [conf 값별 평균 F1]')
print('='*50)
conf_avg = df.groupby('conf')[['precision','recall','f1']].mean().round(4)
print(conf_avg.to_string())

print('\n' + '='*50)
print('  [iou 값별 평균 F1]')
print('='*50)
iou_avg = df.groupby('iou')[['precision','recall','f1']].mean().round(4)
print(iou_avg.to_string())

print('\n' + '='*50)
print('  [imgsz 값별 평균 F1]')
print('='*50)
imgsz_avg = df.groupby('imgsz')[['precision','recall','f1']].mean().round(4)
print(imgsz_avg.to_string())

## STEP 7. 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# conf별 F1
ax = axes[0]
conf_avg['f1'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('conf별 평균 F1 Score', fontsize=13, fontweight='bold')
ax.set_xlabel('conf')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
for i, v in enumerate(conf_avg['f1']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=8)

# iou별 F1
ax = axes[1]
iou_avg['f1'].plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_title('iou별 평균 F1 Score', fontsize=13, fontweight='bold')
ax.set_xlabel('iou')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
for i, v in enumerate(iou_avg['f1']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=8)

# imgsz별 F1
ax = axes[2]
imgsz_avg['f1'].plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='black')
ax.set_title('imgsz별 평균 F1 Score', fontsize=13, fontweight='bold')
ax.set_xlabel('imgsz')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
for i, v in enumerate(imgsz_avg['f1']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('inference_tuning_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✔ 차트 저장 완료: inference_tuning_chart.png')

## STEP 8. Precision vs Recall 트레이드오프 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# conf별로 색상 구분
for conf_val in CONF_RANGE:
    subset = df[df['conf'] == conf_val]
    ax.scatter(
        subset['recall'], subset['precision'],
        label=f'conf={conf_val}',
        s=60, alpha=0.7
    )

# 최적점 강조
ax.scatter(
    best['recall'], best['precision'],
    color='red', s=200, marker='*', zorder=5,
    label=f'★ Best (F1={best["f1"]})'
)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision vs Recall (conf별 분포)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('precision_recall_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✔ 차트 저장 완료: precision_recall_tradeoff.png')

## STEP 9. 전체 결과 CSV 저장 & Project Report 출력

In [ ]:
# CSV 저장
df_sorted.to_csv('inference_tuning_results.csv', encoding='utf-8-sig')
print('✔ 전체 결과 저장: inference_tuning_results.csv')

# Project_Report.md 에 붙일 마크다운 출력
print('\n' + '='*70)
print('  아래 내용을 Project_Report.md 에 복사하세요')
print('='*70)

report_md = f"""
---

## 4. Inference 파라미터 튜닝 (모델 weight 유지)

YOLOv8L 모델의 weight를 변경하지 않고, 추론 시 후처리 파라미터(conf, iou, imgsz)의
최적 조합을 그리드 서치로 탐색하였다.

### 실험 설정

| 항목 | 설정 |
|---|---|
| 모델 | yolov8l.pt (weight 고정) |
| 검증 데이터셋 | COCO128 (Ground Truth 기반) |
| 평가 클래스 | MVP 15 클래스 |
| 실행 환경 | Google Colab GPU (T4) |
| conf 탐색 범위 | {CONF_RANGE} |
| iou 탐색 범위 | {IOU_RANGE} |
| imgsz 탐색 범위 | {IMGSZ_RANGE} |
| 총 조합 수 | {total_combos}개 |

### 최적 파라미터 조합

| 항목 | 기본값 | 최적값 |
|---|---|---|
| conf | 0.25 | **{best['conf']}** |
| iou | 0.7 | **{best['iou']}** |
| imgsz | 640 | **{int(best['imgsz'])}** |
"""

if not baseline.empty:
    bl = baseline.iloc[0]
    f1_diff = best['f1'] - bl['f1']
    report_md += f"""
### 기본값 vs 최적값 비교

| 항목 | 기본값 (conf=0.25, iou=0.7, imgsz=640) | 최적값 (conf={best['conf']}, iou={best['iou']}, imgsz={int(best['imgsz'])}) | 변화 |
|---|---|---|---|
| Precision | {bl['precision']} | **{best['precision']}** | {best['precision'] - bl['precision']:+.4f} |
| Recall | {bl['recall']} | **{best['recall']}** | {best['recall'] - bl['recall']:+.4f} |
| F1 Score | {bl['f1']} | **{best['f1']}** | {f1_diff:+.4f} ({f1_diff/bl['f1']*100:+.1f}%) |
"""

# TOP 5 표
top5 = df_sorted.head(5)
report_md += "\n### TOP 5 파라미터 조합\n\n"
report_md += "| 순위 | conf | iou | imgsz | Precision | Recall | F1 Score |\n"
report_md += "|---|---|---|---|---|---|---|\n"
for rank, (_, row) in enumerate(top5.iterrows(), 1):
    report_md += f"| {rank} | {row['conf']} | {row['iou']} | {int(row['imgsz'])} | {row['precision']} | {row['recall']} | {row['f1']} |\n"

report_md += """
### 파라미터별 영향 분석

- **conf**: 값이 높아질수록 Precision ↑ Recall ↓ (오탐 줄이고 누락 증가)
- **iou**: NMS 중복 제거 강도 조절 (낮으면 겹친 박스 제거, 높으면 보존)
- **imgsz**: 해상도 높을수록 정확도 ↑ 속도 ↓

### 결론

모델 weight 변경 없이 inference 파라미터 튜닝만으로 F1 Score 개선을 확인하였다.
최적 파라미터를 실서비스 파이프라인에 적용하여 광고 객체 탐지 정확도를 높인다.
"""

print(report_md)

## STEP 10. 파일 다운로드

결과 파일을 로컬로 다운로드합니다.

In [ ]:
from google.colab import files

files.download('inference_tuning_results.csv')
files.download('inference_tuning_chart.png')
files.download('precision_recall_tradeoff.png')

print('\n✔ 다운로드 완료!')
print('  - inference_tuning_results.csv    (전체 결과)')
print('  - inference_tuning_chart.png       (파라미터별 F1 차트)')
print('  - precision_recall_tradeoff.png    (Precision-Recall 분포)')